# 06. Topic Modeling & Classification

**Objective:** Discover latent topics in financial news using unsupervised learning

**Methods:**
- **LDA (Latent Dirichlet Allocation)**: Probabilistic topic modeling
- **TF-IDF**: Term frequency analysis for topic characterization
- **Coherence Scoring**: Evaluate topic interpretability
- **Temporal Analysis**: Track topic evolution over time

**Target Topics:**
- Technology & Innovation
- Healthcare & Biotech
- Financial Services & Banking
- Energy & Utilities
- Retail & Consumer


In [6]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# NLP
import re
import string
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.utils import simple_preprocess
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK data
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(" All libraries loaded successfully")

 All libraries loaded successfully


## 1. Setup Paths and Configuration

In [7]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
MODELS_DIR = BASE_DIR / '04_MODELS'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create directories
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

with open(CONFIG_DIR / 'financial_lexicon.json', 'r') as f:
    lexicon = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Features directory: {FEATURES_DIR}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Configuration loaded")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Features directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features
 Results directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS
 Configuration loaded


## 2. Load Event-Enriched Dataset

In [8]:
# Load dataset with events from notebook 05
df = pd.read_csv(FEATURES_DIR / 'dataset_with_events.csv')
df['date'] = pd.to_datetime(df['date'])

print(f" Loaded {len(df):,} articles")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")
print(f"\n Available features:")
print(f"  - Sentiment: finbert_sentiment, vader_compound, textblob_polarity")
print(f"  - Entities: num_entities, num_orgs, num_persons, num_locations")
print(f"  - Events: num_events, primary_event, event_impact_score")

df.head()

 Loaded 63 articles
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19

 Available features:
  - Sentiment: finbert_sentiment, vader_compound, textblob_polarity
  - Entities: num_entities, num_orgs, num_persons, num_locations
  - Events: num_events, primary_event, event_impact_score


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,has_PRODUCT_LAUNCH,has_REGULATORY_ACTION,has_EXECUTIVE_CHANGE,has_LEGAL_ISSUE,has_PARTNERSHIP_DEAL,has_MARKET_DISRUPTION,has_DIVIDEND_ANNOUNCEMENT,has_RESTRUCTURING,has_ANALYST_RATING,has_ECONOMIC_INDICATOR
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,...,0,0,0,0,0,0,0,0,0,0
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0,0,0,0,0,0,0,0,0,0
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,...,0,0,0,0,0,0,0,0,0,0
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,...,0,0,0,0,0,0,0,0,0,0
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,...,0,0,0,0,0,0,0,0,0,0


## 3. Text Preprocessing for Topic Modeling

In [9]:
class TextPreprocessor:
    """Preprocessing pipeline for topic modeling"""
    
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        
        # Add financial stopwords
        financial_stopwords = {
            'said', 'says', 'also', 'would', 'could', 'may', 'might',
            'one', 'two', 'three', 'first', 'second', 'third',
            'reuters', 'bloomberg', 'cnbc', 'wsj', 'financial', 'news',
            'company', 'companies', 'business', 'market', 'markets'
        }
        self.stop_words.update(financial_stopwords)
        
        self.lemmatizer = WordNetLemmatizer()
    
    def preprocess(self, text):
        """Clean and preprocess text"""
        if pd.isna(text) or not isinstance(text, str):
            return []
        
        # Lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)
        
        # Remove email
        text = re.sub(r'\S+@\S+', '', text)
        
        # Remove numbers but keep words with numbers
        text = re.sub(r'\b\d+\b', '', text)
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = simple_preprocess(text, deacc=True)
        
        # Remove stopwords and short tokens
        tokens = [t for t in tokens if t not in self.stop_words and len(t) > 3]
        
        # Lemmatize
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens]
        
        return tokens

# Initialize preprocessor
preprocessor = TextPreprocessor()

# Test preprocessing
test_text = "Apple Inc. reported Q1 earnings of $1.2 billion, beating analyst estimates. The company's stock rose 5% on the news."
test_tokens = preprocessor.preprocess(test_text)
print(f" Test preprocessing:")
print(f"Original: {test_text}")
print(f"Tokens: {test_tokens}")

print("\n Preprocessor initialized")

 Test preprocessing:
Original: Apple Inc. reported Q1 earnings of $1.2 billion, beating analyst estimates. The company's stock rose 5% on the news.
Tokens: ['apple', 'reported', 'earnings', 'billion', 'beating', 'analyst', 'estimate', 'company', 'stock', 'rose']

 Preprocessor initialized


## 4. Preprocess All Documents

In [10]:
print(" Preprocessing all documents...")

# Combine title and text for better context
df['combined_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Preprocess
processed_docs = []
for text in tqdm(df['combined_text'], desc="Processing"):
    tokens = preprocessor.preprocess(text)
    processed_docs.append(tokens)

# Filter out empty documents
valid_indices = [i for i, doc in enumerate(processed_docs) if len(doc) > 5]
processed_docs = [processed_docs[i] for i in valid_indices]
df_filtered = df.iloc[valid_indices].reset_index(drop=True)

print(f"\n Preprocessing complete!")
print(f"  Original documents: {len(df):,}")
print(f"  Valid documents: {len(processed_docs):,}")
print(f"  Average tokens per document: {np.mean([len(d) for d in processed_docs]):.1f}")

# Sample processed document
print(f"\n Sample processed document:")
print(f"Original title: {df_filtered.iloc[0]['title']}")
print(f"Processed tokens: {processed_docs[0][:20]}...")

 Preprocessing all documents...


Processing:   0%|          | 0/63 [00:00<?, ?it/s]


 Preprocessing complete!
  Original documents: 63
  Valid documents: 63
  Average tokens per document: 18.6

 Sample processed document:
Original title: American Express Company $AXP Shares Acquired by Nations Financial Group Inc. IA ADV - MarketBeat
Processed tokens: ['american', 'express', 'share', 'acquired', 'nation', 'group', 'marketbeat', 'american', 'express', 'share', 'acquired', 'nation', 'group']...


## 5. Create Gensim Dictionary and Corpus

In [11]:
# Create dictionary
dictionary = corpora.Dictionary(processed_docs)

print(f" Dictionary statistics (before filtering):")
print(f"  Total unique tokens: {len(dictionary):,}")

# Filter extremes
# no_below: ignore tokens appearing in less than 10 documents
# no_above: ignore tokens appearing in more than 50% of documents
# keep_n: keep only top 10000 most frequent tokens
dictionary.filter_extremes(no_below=10, no_above=0.5, keep_n=10000)

print(f"\n Dictionary statistics (after filtering):")
print(f"  Unique tokens: {len(dictionary):,}")

# Create corpus (Bag of Words)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"\n Corpus statistics:")
print(f"  Number of documents: {len(corpus):,}")
print(f"  Average terms per document: {np.mean([len(doc) for doc in corpus]):.1f}")

# Save dictionary and corpus
dictionary.save(str(MODELS_DIR / 'dictionary.gensim'))
corpora.MmCorpus.serialize(str(MODELS_DIR / 'corpus.mm'), corpus)

print(f"\n Dictionary and corpus saved to {MODELS_DIR}")

 Dictionary statistics (before filtering):
  Total unique tokens: 362

 Dictionary statistics (after filtering):
  Unique tokens: 3

 Corpus statistics:
  Number of documents: 63
  Average terms per document: 0.5

 Dictionary and corpus saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS


## 6. Train LDA Topic Model

In [12]:
# Train LDA model
print(" Training LDA model...")

num_topics = 10
passes = 10
iterations = 100
alpha = 'auto'
eta = 'auto'

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=passes,
    iterations=iterations,
    alpha=alpha,
    eta=eta,
    per_word_topics=True,
    minimum_probability=0.0
)

print(f"\n LDA model trained with {num_topics} topics")

# Save model
lda_model.save(str(MODELS_DIR / 'lda_model.gensim'))
print(f" Model saved to {MODELS_DIR / 'lda_model.gensim'}")

 Training LDA model...

 LDA model trained with 10 topics
 Model saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\lda_model.gensim


## 7. Display Top Topics

In [13]:
# Display top words for each topic
print(" Top 10 Topics with Keywords:")
print("=" * 100)

topic_keywords = []
for topic_id in range(num_topics):
    top_words = lda_model.show_topic(topic_id, topn=10)
    keywords = [word for word, prob in top_words]
    keyword_probs = [prob for word, prob in top_words]
    
    topic_keywords.append({
        'topic_id': topic_id,
        'keywords': ', '.join(keywords[:10]),
        'top_words': keywords,
        'word_probs': keyword_probs
    })
    
    print(f"\nTopic {topic_id}:")
    print(f"  Keywords: {', '.join(keywords[:10])}")
    print(f"  Top words with probabilities:")
    for word, prob in top_words[:5]:
        print(f"    {word}: {prob:.4f}")

# Save topic keywords
topic_keywords_df = pd.DataFrame(topic_keywords)
topic_keywords_df.to_csv(OUTPUTS_DIR / 'topic_keywords.csv', index=False)
print(f"\n Topic keywords saved to {OUTPUTS_DIR / 'topic_keywords.csv'}")

 Top 10 Topics with Keywords:

Topic 0:
  Keywords: price, finance, yahoo
  Top words with probabilities:
    price: 0.5571
    finance: 0.2215
    yahoo: 0.2215

Topic 1:
  Keywords: finance, yahoo, price
  Top words with probabilities:
    finance: 0.6543
    yahoo: 0.3429
    price: 0.0029

Topic 2:
  Keywords: finance, yahoo, price
  Top words with probabilities:
    finance: 0.4663
    yahoo: 0.3325
    price: 0.2012

Topic 3:
  Keywords: finance, yahoo, price
  Top words with probabilities:
    finance: 0.3604
    yahoo: 0.3333
    price: 0.3062

Topic 4:
  Keywords: price, finance, yahoo
  Top words with probabilities:
    price: 0.9895
    finance: 0.0052
    yahoo: 0.0052

Topic 5:
  Keywords: price, finance, yahoo
  Top words with probabilities:
    price: 0.3333
    finance: 0.3333
    yahoo: 0.3333

Topic 6:
  Keywords: price, finance, yahoo
  Top words with probabilities:
    price: 0.3333
    finance: 0.3333
    yahoo: 0.3333

Topic 7:
  Keywords: price, finance, yahoo
  

## 8. Calculate Topic Coherence

In [14]:
# Calculate coherence score
print(" Calculating topic coherence...")

coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print(f"\n Model Evaluation:")
print(f"  Coherence Score (c_v): {coherence_score:.4f}")
print(f"  Number of topics: {num_topics}")
print(f"  Perplexity: {lda_model.log_perplexity(corpus):.4f}")

if coherence_score > 0.45:
    print(f"   EXCELLENT - Exceeds target (>0.45)")
elif coherence_score > 0.40:
    print(f"   GOOD - Meets industry benchmark (>0.40)")
else:
    print(f"   NEEDS IMPROVEMENT - Below benchmark")

# Save metrics
metrics = {
    'num_topics': num_topics,
    'coherence_score': float(coherence_score),
    'perplexity': float(lda_model.log_perplexity(corpus)),
    'num_documents': len(corpus),
    'vocabulary_size': len(dictionary)
}

with open(OUTPUTS_DIR / 'topic_model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n Metrics saved to {OUTPUTS_DIR / 'topic_model_metrics.json'}")

 Calculating topic coherence...

 Model Evaluation:
  Coherence Score (c_v): 0.6360
  Number of topics: 10
  Perplexity: -1.5735
   EXCELLENT - Exceeds target (>0.45)

 Metrics saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\topic_model_metrics.json


## 9. Assign Topics to Documents

In [15]:
# Get dominant topic for each document
print(" Assigning topics to documents...")

dominant_topics = []
topic_distributions = []

for doc_bow in tqdm(corpus, desc="Assigning topics"):
    # Get topic distribution for document
    topic_dist = lda_model.get_document_topics(doc_bow)
    
    # Sort by probability
    topic_dist_sorted = sorted(topic_dist, key=lambda x: x[1], reverse=True)
    
    # Get dominant topic
    if len(topic_dist_sorted) > 0:
        dominant_topic = topic_dist_sorted[0][0]
        dominant_prob = topic_dist_sorted[0][1]
    else:
        dominant_topic = -1
        dominant_prob = 0.0
    
    dominant_topics.append(dominant_topic)
    topic_distributions.append(topic_dist)

# Add to dataframe
df_filtered['dominant_topic'] = dominant_topics
df_filtered['topic_distribution'] = topic_distributions

# Add topic keywords
df_filtered['topic_keywords'] = df_filtered['dominant_topic'].apply(
    lambda x: topic_keywords_df.iloc[x]['keywords'] if x >= 0 else 'UNKNOWN'
)

print(f"\n Topics assigned to {len(df_filtered):,} documents")
print(f"\n Topic distribution:")
print(df_filtered['dominant_topic'].value_counts().sort_index())

 Assigning topics to documents...


Assigning topics:   0%|          | 0/63 [00:00<?, ?it/s]


 Topics assigned to 63 documents

 Topic distribution:
dominant_topic
1    54
4     9
Name: count, dtype: int64


## 10. Visualize Topic Distribution

In [16]:
# Topic distribution bar chart
topic_counts = df_filtered['dominant_topic'].value_counts().sort_index()

# Add topic keywords to labels
topic_labels = []
for topic_id in topic_counts.index:
    keywords = topic_keywords_df.iloc[topic_id]['top_words'][:3]
    label = f"Topic {topic_id}: {', '.join(keywords)}"
    topic_labels.append(label)

fig = go.Figure(data=[
    go.Bar(
        x=topic_labels,
        y=topic_counts.values,
        marker=dict(
            color=topic_counts.values,
            colorscale='Viridis',
            showscale=True
        ),
        text=topic_counts.values,
        textposition='auto'
    )
])

fig.update_layout(
    title="Document Distribution Across Topics",
    xaxis_title="Topic",
    yaxis_title="Number of Documents",
    height=500,
    xaxis_tickangle=-45
)

fig.write_html(VIZ_DIR / 'topic_distribution.html')
fig.show()

print(" Topic distribution visualization saved")

 Topic distribution visualization saved


## 11. Topic Evolution Over Time

In [17]:
# Track topics over time
df_filtered['year_month'] = df_filtered['date'].dt.to_period('M').astype(str)

# Monthly topic distribution
topic_timeline = df_filtered.groupby(['year_month', 'dominant_topic']).size().reset_index(name='count')

# Plot evolution of each topic
fig = px.line(
    topic_timeline,
    x='year_month',
    y='count',
    color='dominant_topic',
    title='Topic Evolution Over Time',
    labels={'year_month': 'Month', 'count': 'Number of Articles', 'dominant_topic': 'Topic ID'}
)

fig.update_layout(
    height=600,
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig.write_html(VIZ_DIR / 'topic_evolution.html')
fig.show()

# Stacked area chart
topic_timeline_pivot = topic_timeline.pivot(index='year_month', columns='dominant_topic', values='count').fillna(0)

fig2 = go.Figure()
for topic_id in topic_timeline_pivot.columns:
    keywords = topic_keywords_df.iloc[topic_id]['top_words'][:3]
    fig2.add_trace(go.Scatter(
        x=topic_timeline_pivot.index,
        y=topic_timeline_pivot[topic_id],
        mode='lines',
        stackgroup='one',
        name=f"Topic {topic_id}: {', '.join(keywords)}"
    ))

fig2.update_layout(
    title='Topic Distribution Over Time (Stacked)',
    xaxis_title='Month',
    yaxis_title='Number of Articles',
    height=600,
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig2.write_html(VIZ_DIR / 'topic_evolution_stacked.html')
fig2.show()

print(" Temporal analysis complete")

 Temporal analysis complete


## 12. Topic-Sentiment Analysis

In [18]:
# Analyze sentiment by topic
topic_sentiment = df_filtered.groupby('dominant_topic').agg({
    'finbert_sentiment': 'mean',
    'vader_compound': 'mean',
    'textblob_polarity': 'mean',
    'event_impact_score': 'mean'
}).reset_index()

# Add topic keywords
topic_sentiment['keywords'] = topic_sentiment['dominant_topic'].apply(
    lambda x: ', '.join(topic_keywords_df.iloc[x]['top_words'][:3])
)

print(" Average Sentiment by Topic:")
print("=" * 100)
print(f"{'Topic':8s} {'Keywords':40s} {'FinBERT':>10s} {'VADER':>10s} {'Impact':>10s}")
print("=" * 100)
for idx, row in topic_sentiment.iterrows():
    print(f"{int(row['dominant_topic']):8d} {row['keywords']:40s} "
          f"{row['finbert_sentiment']:10.3f} {row['vader_compound']:10.3f} "
          f"{row['event_impact_score']:10.3f}")

# Visualize
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"Topic {int(row['dominant_topic'])}: {row['keywords'][:30]}" for _, row in topic_sentiment.iterrows()],
    y=topic_sentiment['finbert_sentiment'],
    name='FinBERT Sentiment',
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    x=[f"Topic {int(row['dominant_topic'])}: {row['keywords'][:30]}" for _, row in topic_sentiment.iterrows()],
    y=topic_sentiment['event_impact_score'],
    name='Event Impact',
    marker_color='coral'
))

fig.update_layout(
    title='Sentiment and Impact by Topic',
    xaxis_title='Topic',
    yaxis_title='Score',
    height=600,
    xaxis_tickangle=-45,
    barmode='group'
)

fig.write_html(VIZ_DIR / 'topic_sentiment_analysis.html')
fig.show()

# Save
topic_sentiment.to_csv(OUTPUTS_DIR / 'topic_sentiment_analysis.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'topic_sentiment_analysis.csv'}")

 Average Sentiment by Topic:
Topic    Keywords                                    FinBERT      VADER     Impact
       1 finance, yahoo, price                         0.124      0.124      0.008
       4 price, finance, yahoo                         0.064      0.064      0.000



 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\topic_sentiment_analysis.csv


## 13. Interactive Topic Visualization (pyLDAvis)

In [19]:
# Create interactive visualization
print(" Creating interactive topic visualization...")

try:
    vis = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False)
    pyLDAvis.save_html(vis, str(VIZ_DIR / 'topic_model_interactive.html'))
    print(f" Interactive visualization saved to {VIZ_DIR / 'topic_model_interactive.html'}")
    print("   Open this HTML file in a browser for interactive exploration")
except Exception as e:
    print(f" Could not create pyLDAvis visualization: {e}")
    print("   This is optional and doesn't affect the analysis")

 Creating interactive topic visualization...
 Interactive visualization saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\topic_model_interactive.html
   Open this HTML file in a browser for interactive exploration


## 14. Topic-Company Analysis

In [20]:
# Analyze which topics are most common for each company
company_topics = df_filtered.groupby(['ticker', 'dominant_topic']).size().reset_index(name='count')

# Get dominant topic for top companies
top_companies = df_filtered['ticker'].value_counts().head(20).index
company_topic_profiles = []

for ticker in top_companies:
    ticker_topics = company_topics[company_topics['ticker'] == ticker].sort_values('count', ascending=False)
    
    if len(ticker_topics) > 0:
        dominant_topic = ticker_topics.iloc[0]['dominant_topic']
        topic_keywords = topic_keywords_df.iloc[dominant_topic]['keywords']
        
        company_topic_profiles.append({
            'ticker': ticker,
            'dominant_topic': dominant_topic,
            'topic_keywords': topic_keywords,
            'article_count': ticker_topics['count'].sum()
        })

company_topic_df = pd.DataFrame(company_topic_profiles)

print(" Topic Profiles for Top 20 Companies:")
print("=" * 100)
for idx, row in company_topic_df.iterrows():
    print(f"{row['ticker']:8s} | Topic {int(row['dominant_topic'])}: {row['topic_keywords'][:60]}")

# Save
company_topic_df.to_csv(OUTPUTS_DIR / 'company_topic_profiles.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'company_topic_profiles.csv'}")

 Topic Profiles for Top 20 Companies:
ABT      | Topic 1: finance, yahoo, price
AVGO     | Topic 1: finance, yahoo, price
BABA     | Topic 1: finance, yahoo, price
BAC      | Topic 1: finance, yahoo, price
APA      | Topic 1: finance, yahoo, price
AMD      | Topic 1: finance, yahoo, price
BA       | Topic 1: finance, yahoo, price
AVB      | Topic 1: finance, yahoo, price
AZN      | Topic 1: finance, yahoo, price
ACN      | Topic 1: finance, yahoo, price
AMT      | Topic 1: finance, yahoo, price
AAPL     | Topic 1: finance, yahoo, price
ASML     | Topic 1: finance, yahoo, price
AMZN     | Topic 1: finance, yahoo, price
ADBE     | Topic 1: finance, yahoo, price
AMGN     | Topic 1: finance, yahoo, price
BBD      | Topic 1: finance, yahoo, price
AXP      | Topic 1: finance, yahoo, price
AEP      | Topic 1: finance, yahoo, price

 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\company_topic_profiles.csv


## 15. Save Enhanced Dataset

In [22]:
# Helper function to convert numpy types to Python types for JSON serialization
def convert_topic_dist_to_json(topic_dist):
    """Convert topic distribution with numpy types to JSON-serializable format"""
    return json.dumps([(int(topic_id), float(prob)) for topic_id, prob in topic_dist])

# Convert topic distribution to JSON string
df_filtered['topic_distribution_json'] = df_filtered['topic_distribution'].apply(convert_topic_dist_to_json)

# Create probability columns for each topic
for topic_id in range(num_topics):
    df_filtered[f'topic_{topic_id}_prob'] = df_filtered['topic_distribution'].apply(
        lambda x: float(next((prob for tid, prob in x if tid == topic_id), 0.0))
    )

# Save to CSV (drop original list columns)
df_to_save = df_filtered.drop(columns=['combined_text', 'topic_distribution'])
df_to_save.to_csv(FEATURES_DIR / 'dataset_with_topics.csv', index=False)

print(f"\n Saved enhanced dataset to {FEATURES_DIR / 'dataset_with_topics.csv'}")
print(f"Dataset shape: {df_to_save.shape}")

print(f"  - topic_0_prob, topic_1_prob, ..., topic_{num_topics-1}_prob")

df_to_save.head()


 Saved enhanced dataset to c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_topics.csv
Dataset shape: (63, 60)
  - topic_0_prob, topic_1_prob, ..., topic_9_prob


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,...,0.084952,0.178254,0.087264,0.083042,0.153865,0.081756,0.081756,0.081756,0.085599,0.081756
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.027539,0.057783,0.028288,0.026919,0.725714,0.026502,0.026502,0.026502,0.027748,0.026502
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,...,0.020583,0.800899,0.021143,0.020120,0.037280,0.019809,0.019809,0.019809,0.020740,0.019809
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,...,0.084952,0.178254,0.087264,0.083042,0.153865,0.081756,0.081756,0.081756,0.085599,0.081756
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,...,0.084952,0.178254,0.087264,0.083042,0.153865,0.081756,0.081756,0.081756,0.085599,0.081756


## 16. Summary Statistics

In [23]:
# Create comprehensive summary
summary = {
    'total_documents': len(df_filtered),
    'num_topics': num_topics,
    'coherence_score': float(coherence_score),
    'perplexity': float(lda_model.log_perplexity(corpus)),
    'vocabulary_size': len(dictionary),
    'avg_document_length': float(np.mean([len(d) for d in processed_docs])),
    'topic_distribution': dict(df_filtered['dominant_topic'].value_counts().sort_index()),
    'topic_keywords': topic_keywords_df.to_dict('records'),
    'date_range': {
        'start': str(df_filtered['date'].min()),
        'end': str(df_filtered['date'].max())
    },
    'model_parameters': {
        'passes': passes,
        'iterations': iterations,
        'alpha': alpha,
        'eta': eta
    }
}

# Save summary
with open(OUTPUTS_DIR / 'topic_modeling_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Topic Modeling & Classification Summary")
print("=" * 70)
print(f"Total documents processed: {summary['total_documents']:,}")
print(f"Number of topics: {summary['num_topics']}")
print(f"Coherence score: {summary['coherence_score']:.4f}")
print(f"Vocabulary size: {summary['vocabulary_size']:,}")
print(f"Average document length: {summary['avg_document_length']:.1f} tokens")

print(f"\n Summary saved to {OUTPUTS_DIR / 'topic_modeling_summary.json'}")
print("\n Topic Modeling & Classification complete!")
print("\n Output files:")
print(f"  - {FEATURES_DIR / 'dataset_with_topics.csv'}")
print(f"  - {OUTPUTS_DIR / 'topic_keywords.csv'}")
print(f"  - {OUTPUTS_DIR / 'topic_sentiment_analysis.csv'}")
print(f"  - {OUTPUTS_DIR / 'company_topic_profiles.csv'}")
print(f"  - {MODELS_DIR / 'lda_model.gensim'}")
print(f"  - {MODELS_DIR / 'dictionary.gensim'}")
print(f"  - {VIZ_DIR / 'topic_distribution.html'}")
print(f"  - {VIZ_DIR / 'topic_evolution.html'}")
print(f"  - {VIZ_DIR / 'topic_sentiment_analysis.html'}")
print(f"  - {VIZ_DIR / 'topic_model_interactive.html'}")

 Topic Modeling & Classification Summary
Total documents processed: 63
Number of topics: 10
Coherence score: 0.6360
Vocabulary size: 3
Average document length: 18.6 tokens

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\topic_modeling_summary.json

 Topic Modeling & Classification complete!

 Output files:
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_topics.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\topic_keywords.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\topic_sentiment_analysis.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\company_topic_profiles.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\lda_model.gensim
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\04_MODELS\dictionary.gensim
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\topic_distribution.html
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\topic_